# 预习 1：机器学习 / 深度学习基础复习

> 建议放在你现有 Day1 之前学习。  
> 目标不是把传统机器学习全部学完，而是让你在进入 Attention、KV Cache、RoPE、MoE、LoRA 之前，先建立一套稳定的“模型—数据—损失—优化—泛化”心智模型。

---

## 这份 notebook 解决什么问题？

你现有 7 天笔记从 **Scaled Dot-Product Attention** 开始，已经属于现代大模型核心机制。  
但如果没有下面这些前置概念，后面容易变成“只会记公式，不知道公式为什么存在”：

1. **AI / ML / DL / LLM 的层级关系**
2. **监督学习、无监督学习、自监督学习、强化学习**分别在解决什么问题
3. **数据集划分、特征、标签、训练集、验证集、测试集**的工程意义
4. **损失函数、梯度下降、过拟合、正则化、泛化能力**
5. **向量、矩阵、张量、batch、维度**这些你后面看 Attention 必须秒懂的概念

---

## 和你后续 7 天的衔接

| 后续内容 | 这份预习对应的基础 |
|---|---|
| Day1 Attention | 向量点积、softmax、矩阵乘法、batch 维度 |
| Day2 MHA / GQA / KV Cache | 张量形状、参数量、显存估算 |
| Day3 RoPE | 向量空间、位置表示、矩阵变换 |
| Day4 自回归生成 | 概率建模、条件概率、序列预测 |
| Day5 PagedAttention | 训练/推理差异、内存瓶颈 |
| Day6 MoE | 参数量、激活参数、路由分类 |
| Day7 LoRA / RAG / Agent | 微调、检索、嵌入、泛化 |

# 0. 环境准备

本 notebook 只使用：

- `numpy`：做矩阵和梯度计算
- `matplotlib`：画图

不依赖 PyTorch，不依赖 sklearn。  
因为这一阶段的目标是理解原理，而不是先上深度学习框架。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("NumPy version:", np.__version__)

# 1. AI / ML / DL / LLM 的层级关系

先把概念摆正：

```text
人工智能 AI
└── 机器学习 ML
    ├── 传统机器学习
    │   ├── 线性回归 / 逻辑回归
    │   ├── SVM
    │   ├── 决策树 / 随机森林 / XGBoost
    │   └── KMeans / PCA
    │
    └── 深度学习 DL
        ├── MLP
        ├── CNN
        ├── RNN / LSTM / GRU
        ├── GNN
        └── Transformer
            └── 大语言模型 LLM
```

## 关键理解

**AI** 是目标：让机器表现出智能行为。  
**ML** 是方法：让模型从数据中学习规律。  
**DL** 是 ML 的一个分支：用多层神经网络自动学习表示。  
**LLM** 是 DL + Transformer + 大规模数据 + 大规模算力的结果。

所以：

> LLM 不是凭空出现的，它是深度学习发展到 Transformer 之后，在规模化训练上的一个阶段性结果。

# 2. 机器学习到底在学什么？

机器学习的核心抽象可以写成：

$$
\hat{y} = f_\theta(x)
$$

其中：

- $x$：输入，例如图片、文本、表格特征
- $y$：真实标签，例如类别、数值、下一个 token
- $\hat{y}$：模型预测
- $f_\theta$：带参数的模型
- $\theta$：模型参数
- 学习：调整 $\theta$，让 $\hat{y}$ 尽量接近 $y$

更完整的训练目标是：

$$
\theta^\* = \arg\min_\theta \mathcal{L}(f_\theta(x), y)
$$

也就是：找到一组参数，让损失函数尽可能小。

# 3. 监督 / 无监督 / 自监督 / 强化学习

## 3.1 监督学习 Supervised Learning

训练数据有明确标签：

```text
输入 x: 一张猫的图片
标签 y: 猫
```

典型任务：

- 分类：猫/狗/车/人
- 回归：房价预测、温度预测、电池 SOH 预测
- 序列标注：词性标注、命名实体识别

## 3.2 无监督学习 Unsupervised Learning

训练数据没有人工标签，模型自己找结构。

典型任务：

- 聚类：KMeans
- 降维：PCA、t-SNE、UMAP
- 异常检测

## 3.3 自监督学习 Self-Supervised Learning

标签来自数据本身，不需要人工标注。

大语言模型最重要的训练目标之一就是：

> 给定前面的 token，预测下一个 token。

$$
P(x_t \mid x_1, x_2, ..., x_{t-1})
$$

这就是后面 Day4 自回归生成的基础。

## 3.4 强化学习 Reinforcement Learning

智能体在环境中行动，根据奖励学习策略。

```text
状态 state -> 动作 action -> 奖励 reward -> 新状态
```

LLM 中常见的 RLHF / RLAIF，本质上就是把人类偏好或 AI 偏好变成奖励信号。

# 4. 数据集划分：训练集、验证集、测试集

这是机器学习工程里最容易被初学者忽略，但非常关键的部分。

| 数据集 | 作用 | 能不能用来调模型 |
|---|---|---|
| 训练集 train | 更新参数 | 可以 |
| 验证集 validation | 选择超参数、早停、模型选择 | 可以间接使用 |
| 测试集 test | 最终客观评估 | 不应该反复调 |

## 最重要原则：防止数据泄漏

数据泄漏是指：

> 模型在训练阶段直接或间接看到了验证集/测试集的信息，导致评估结果虚高。

常见泄漏：

1. 先对全数据归一化，再划分训练/测试
2. 时间序列随机打乱切分，导致未来信息泄漏到过去
3. 文本去重没做好，训练集和测试集有高度相似样本
4. 用测试集反复调参

后面你做论文、模型训练、RAG 评估时，这个原则都非常重要。

# 5. 向量、矩阵、张量：后面看 Attention 的地基

## 5.1 标量 scalar

一个数字：

$$
3.14
$$

## 5.2 向量 vector

一排数字：

$$
x = [1.2, 0.7, -0.4]
$$

在模型里，一个 token 可以被表示成一个向量。

## 5.3 矩阵 matrix

二维表格：

$$
X \in \mathbb{R}^{T \times d}
$$

可以理解为：

- $T$：序列长度，有多少个 token
- $d$：每个 token 的 embedding 维度

## 5.4 张量 tensor

三维或更高维数组。  
Transformer 里经常看到：

$$
X \in \mathbb{R}^{B \times T \times d}
$$

其中：

- $B$：batch size
- $T$：sequence length
- $d$：hidden dimension

In [ ]:
# 标量、向量、矩阵、张量的 NumPy 形状

scalar = np.array(3.14)
vector = np.array([1.2, 0.7, -0.4])
matrix = np.random.randn(4, 3)      # 4 个 token，每个 token 3 维
tensor = np.random.randn(2, 4, 3)   # batch=2, token=4, dim=3

print("scalar shape:", scalar.shape)
print("vector shape:", vector.shape)
print("matrix shape:", matrix.shape)
print("tensor shape:", tensor.shape)

# 6. 点积：Attention 公式里的第一块砖

两个向量的点积：

$$
a \cdot b = \sum_i a_i b_i
$$

直觉：

- 方向相近：点积大
- 方向垂直：点积接近 0
- 方向相反：点积小或负

后面 Attention 里的：

$$
QK^T
$$

本质就是让每个 query 向量和每个 key 向量做点积，得到“相关性分数”。

In [ ]:
a = np.array([1, 2, 3])
b = np.array([2, 4, 6])
c = np.array([-2, -4, -6])
d = np.array([3, -3, 1])

print("a · b =", np.dot(a, b), "方向很相近")
print("a · c =", np.dot(a, c), "方向相反")
print("a · d =", np.dot(a, d), "相关性较弱")

# 7. 从线性回归理解“参数学习”

线性回归假设：

$$
\hat{y} = wx + b
$$

损失函数使用均方误差：

$$
\mathcal{L} = \frac{1}{N}\sum_i(\hat{y}_i - y_i)^2
$$

训练的目的就是不断调整 $w,b$，让预测线越来越贴近真实数据。

In [ ]:
# 生成一组线性数据
np.random.seed(0)
N = 80
x = np.linspace(-3, 3, N)
true_w, true_b = 2.5, -0.7
noise = np.random.randn(N) * 0.8
y = true_w * x + true_b + noise

# 初始化参数
w = np.random.randn()
b = np.random.randn()
lr = 0.03
loss_history = []

for step in range(300):
    y_pred = w * x + b
    loss = np.mean((y_pred - y) ** 2)
    loss_history.append(loss)

    # 对 w 和 b 求梯度
    grad_w = np.mean(2 * (y_pred - y) * x)
    grad_b = np.mean(2 * (y_pred - y))

    # 梯度下降更新参数
    w -= lr * grad_w
    b -= lr * grad_b

print(f"learned w={w:.3f}, b={b:.3f}")
print(f"true    w={true_w:.3f}, b={true_b:.3f}")

plt.figure(figsize=(7, 4))
plt.scatter(x, y, s=18, label="data")
plt.plot(x, w*x + b, label="learned line")
plt.title("Linear Regression Learned by Gradient Descent")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(loss_history)
plt.title("Training Loss")
plt.xlabel("step")
plt.ylabel("MSE")
plt.show()

# 8. 梯度下降：深度学习训练的核心机制

梯度下降的形式：

$$
\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}
$$

其中：

- $\theta$：参数
- $\eta$：学习率 learning rate
- $\nabla_\theta \mathcal{L}$：损失函数对参数的梯度

## 一句话理解

> 梯度告诉你“损失上升最快的方向”，所以我们沿着反方向走，让损失下降。

## 学习率太大会怎样？

- loss 震荡
- 甚至发散

## 学习率太小会怎样？

- 训练非常慢
- 可能卡在不好的区域

In [ ]:
def run_gd(lr, steps=40):
    w = -5.0
    history = []
    for _ in range(steps):
        # 简单函数：L=(w-3)^2
        loss = (w - 3) ** 2
        grad = 2 * (w - 3)
        w -= lr * grad
        history.append(loss)
    return np.array(history)

plt.figure(figsize=(7, 4))
for lr in [0.01, 0.1, 0.7, 1.1]:
    hist = run_gd(lr)
    plt.plot(hist, label=f"lr={lr}")
plt.ylim(0, 100)
plt.title("Learning Rate Comparison")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

# 9. 分类任务：从 sigmoid 到 softmax

## 9.1 二分类 sigmoid

二分类常用 sigmoid：

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

它把任意实数压到 0 到 1 之间，可以解释成概率。

## 9.2 多分类 softmax

多分类常用 softmax：

$$
\text{softmax}(z_i)=\frac{e^{z_i}}{\sum_j e^{z_j}}
$$

它把一组 logits 转成概率分布。

后面 Attention 里也会用 softmax，不过它不是用来分类，而是把 attention score 转成 attention weight。

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def softmax(z):
    z = z - np.max(z)  # 数值稳定
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z)

z = np.linspace(-8, 8, 200)
plt.figure(figsize=(7, 4))
plt.plot(z, sigmoid(z))
plt.title("Sigmoid Function")
plt.xlabel("z")
plt.ylabel("sigmoid(z)")
plt.show()

logits = np.array([1.2, 0.3, -0.5, 2.1])
probs = softmax(logits)

print("logits:", logits)
print("softmax probabilities:", probs)
print("sum:", probs.sum())

# 10. 交叉熵损失：分类模型为什么这么训练？

对于真实类别 $y$，模型预测概率为 $p_y$，交叉熵损失为：

$$
\mathcal{L} = -\log p_y
$$

如果模型给真实类别很高概率，损失很小。  
如果模型给真实类别很低概率，损失很大。

## 举例

真实类别是第 2 类：

- 模型 A：`[0.05, 0.90, 0.05]`，损失小
- 模型 B：`[0.80, 0.10, 0.10]`，损失大

这就是语言模型训练“预测下一个 token”的核心损失。

In [ ]:
def cross_entropy(probs, target_index):
    return -np.log(probs[target_index] + 1e-12)

p_good = np.array([0.05, 0.90, 0.05])
p_bad = np.array([0.80, 0.10, 0.10])

print("good prediction CE:", cross_entropy(p_good, 1))
print("bad prediction CE:", cross_entropy(p_bad, 1))

# 11. 过拟合与泛化

## 11.1 什么是过拟合？

模型在训练集上表现很好，但在新数据上表现很差。

```text
训练集准确率：99%
测试集准确率：62%
```

这通常说明模型记住了训练数据的细节，而不是学到稳定规律。

## 11.2 过拟合常见原因

1. 模型太复杂，数据太少
2. 训练太久
3. 数据噪声太大
4. 数据划分泄漏或分布不一致
5. 正则化不足

## 11.3 常见缓解方法

- 更多数据
- 数据增强
- L2 正则化 / weight decay
- dropout
- early stopping
- 减小模型规模
- 更合理的数据划分

In [ ]:
# 用多项式拟合展示欠拟合、合适拟合、过拟合
np.random.seed(1)
x = np.linspace(-1, 1, 25)
y = np.sin(3 * x) + np.random.randn(len(x)) * 0.15

x_test = np.linspace(-1, 1, 200)
y_true = np.sin(3 * x_test)

degrees = [1, 3, 12]
plt.figure(figsize=(9, 4))

for deg in degrees:
    coef = np.polyfit(x, y, deg)
    y_fit = np.polyval(coef, x_test)
    plt.plot(x_test, y_fit, label=f"degree={deg}")

plt.scatter(x, y, s=25, label="train data")
plt.plot(x_test, y_true, linestyle="--", label="true function")
plt.title("Underfitting vs Good Fit vs Overfitting")
plt.legend()
plt.show()

# 12. 归一化：为什么深度学习常常要标准化输入？

很多模型对输入尺度敏感。

例如两个特征：

```text
年龄：0 到 100
收入：0 到 1,000,000
```

如果不归一化，收入这个特征会在数值上压倒年龄。

常见方法：

## 标准化 Standardization

$$
x'=\frac{x-\mu}{\sigma}
$$

## Min-Max Scaling

$$
x'=\frac{x-x_{\min}}{x_{\max}-x_{\min}}
$$

## 重要工程原则

标准化参数必须只在训练集上估计，然后应用到验证集和测试集。  
不能用全数据一起算均值和方差，否则会泄漏测试集信息。

In [ ]:
train = np.array([1, 2, 3, 4, 5], dtype=float)
test = np.array([100, 101], dtype=float)

# 正确：只用 train 的均值方差
mu_train = train.mean()
std_train = train.std()

train_scaled = (train - mu_train) / std_train
test_scaled = (test - mu_train) / std_train

print("train mean/std:", mu_train, std_train)
print("train_scaled:", train_scaled)
print("test_scaled:", test_scaled)

# 错误示例：把 train + test 一起算，会让测试集信息进入训练流程
all_data = np.concatenate([train, test])
print("wrong global mean/std:", all_data.mean(), all_data.std())

# 13. 模型分类速查表

| 类别 | 代表模型 | 主要用途 |
|---|---|---|
| 线性模型 | Linear Regression, Logistic Regression | 简单、可解释、强 baseline |
| 树模型 | Decision Tree, Random Forest, XGBoost | 表格数据强，特征解释常用 |
| 聚类模型 | KMeans, DBSCAN | 无标签分组 |
| 降维模型 | PCA, t-SNE, UMAP | 可视化、压缩、去噪 |
| 神经网络 | MLP, CNN, RNN, Transformer | 图像、文本、语音、多模态 |
| 生成模型 | VAE, GAN, Diffusion, LLM | 图像生成、文本生成、数据生成 |
| 图模型 | GCN, GraphSAGE, GAT | 社交网络、分子图、推荐系统 |

## 初学者建议

不要一上来就觉得“传统机器学习没用”。  
在工程和论文里，传统模型常常是非常重要的 baseline。

# 14. 大模型之前必须掌握的 10 个关键词

1. **参数 parameter**：模型可学习的数字
2. **超参数 hyperparameter**：学习率、batch size、层数等人为设置
3. **embedding**：把离散对象变成连续向量
4. **logits**：softmax 前的原始分数
5. **loss**：训练优化目标
6. **gradient**：损失对参数的变化方向
7. **optimizer**：用梯度更新参数的方法
8. **batch**：一次并行处理的一组样本
9. **epoch**：完整看完一遍训练集
10. **generalization**：模型对未见数据的表现

后面 Attention 里的每一个公式，本质上都逃不开这些概念。

# 15. 自测题

请在进入下一份预习 notebook 前，先尝试回答：

1. AI、ML、DL、LLM 的关系是什么？
2. 监督学习和自监督学习最大的区别是什么？
3. 为什么测试集不能反复用来调参？
4. 什么是数据泄漏？举两个例子。
5. 点积为什么可以衡量两个向量的相关性？
6. softmax 的作用是什么？
7. 交叉熵为什么适合分类任务？
8. 什么是过拟合？如何缓解？
9. 为什么归一化参数不能用全数据计算？
10. Transformer 里的 `[B, T, d]` 分别代表什么？

如果这些问题能基本答出来，就可以进入“预习 2：神经网络架构地图”。